In [ ]:
!nvidia-smi

Fri May 15 21:57:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls

drive  sample_data


In [ ]:
![ -d AlphaZero ] && rm -rf AlphaZero
!git clone https://github.com/NeoAcar/AlphaZero.git
%cd /content/AlphaZero
!ls

Cloning into 'AlphaZero'...
remote: Enumerating objects: 261, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 261 (delta 150), reused 168 (delta 92), pack-reused 0 (from 0)
Receiving objects: 100% (261/261), 177.12 KiB | 2.81 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/AlphaZero
alphazero	  gen_sf_data.py  pyproject.toml  selfplay.py	     uci.py
alphazero_uci.sh  LICENSE	  README.md	  temporary.py	     uv.lock
CLAUDE.md	  match.py	  RECIPE.md	  train_config.json
configs		  play.py	  runner.py	  train.py


In [ ]:
!mkdir -p data/sf_shards_v2
!cp -r /content/drive/MyDrive/alphazero/sf_shards_v2/* data/sf_shards_v2/
!du -sh data/sf_shards_v2/   # sanity check

38G	data/sf_shards_v2/


In [ ]:
!pip install -q 'python-chess>=1.11' 'tensorboard>=2.18' 'tqdm>=4.66'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 62.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eminacar (eminacar-itu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!python train.py \
      --shards-dir /content/AlphaZero/data/sf_shards_v2 \
      --epochs 5 --batch-size 2048 --learning-rate 1e-4 \
      --label-smoothing 0.1 --val-fraction 0.2 --vals-per-epoch 1 \
      --value-head wdl \
      --checkpoint-dir /content/checkpoints/wdl_v1 \
      --log-dir /content/logs/wdl_v1 \
      --wandb-project alphazero-chess --wandb-name wdl_v1

Training config:
{
  "batch_size": 2048,
  "learning_rate": 0.0001,
  "l2_weight": 0.0001,
  "epochs": 5,
  "log_step": 50,
  "label_smoothing": 0.1,
  "checkpoint_dir": "/content/checkpoints/wdl_v1",
  "log_dir": "/content/logs/wdl_v1",
  "val_fraction": 0.2,
  "split_seed": 137,
  "resume": null,
  "self_play_data": null,
  "data_mix": null,
  "wandb_project": "alphazero-chess",
  "wandb_group": null,
  "wandb_name": "wdl_v1",
  "shards_dir": "/content/AlphaZero/data/sf_shards_v2",
  "max_shards": null,
  "value_head": "wdl",
  "vals_per_epoch": 1
}
Model: SEResNetWDL (value_head=wdl)
Loading 56 shard(s) from /content/AlphaZero/data/sf_shards_v2
  shard_0000.pt: 436203 positions, 5000 games
  shard_0001.pt: 437534 positions, 5000 games
  shard_0002.pt: 426542 positions, 5000 games
  shard_0003.pt: 432511 positions, 5000 games
  shard_0004.pt: 434272 positions, 5000 games
  shard_0005.pt: 427320 positions, 5000 games
  shard_0006.pt: 145846 positions, 1666 games
  shard_0000.pt: 44045

In [ ]:
import os

# Create the destination directory in Drive if it doesn't exist
drive_path = '/content/drive/MyDrive/alphazero/checkpoints/wdl_v1'
!mkdir -p "{drive_path}"

# Copy the checkpoints
print(f"Copying checkpoints to {drive_path}...")
!cp -r /content/checkpoints/wdl_v1/* "{drive_path}/"
print("Backup complete.")

Copying checkpoints to /content/drive/MyDrive/alphazero/checkpoints/wdl_v1...
Backup complete.
